# Agent Step Demo

## Installation

follow our [installation document](https://internrobotics.github.io/user_guide/internnav/quick_start/installation.html), install the model by

```
conda create -n <env> python=3.10 libxcb=1.14
pip install torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu118
pip install -e .[model]
```

The kernel that running this notebook has installed the conda environment.

## Start the model server

In [3]:
import subprocess, time

# Start your server as a background process
server = subprocess.Popen(
    ["python", "scripts/eval/start_server.py"],
    text=True,
    cwd="/root/InternNav"
)

print(f"✅ Server started with PID {server.pid}")
time.sleep(3)  # wait a bit for server startup


✅ Server started with PID 22710
PROJECT_ROOT_PATH:/root/InternNav
Starting Agent Server...
Registering agents...


/root/InternNav/./internnav/model/basemodel/LongCLIP/model/longclip.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import packaging
xFormers not available
xFormers not available
INFO:     Started server process [22710]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://localhost:8087 (Press CTRL+C to quit)


## Initialize agent

In [ ]:
import sys
sys.path.insert(0, '/root/InternNav')

from internnav.configs.agent import AgentCfg

agent_cfg=AgentCfg(
    server_port=8087,
    model_name='internvla_n1',
    ckpt_path='',
    model_settings={
        'policy_name': "InternVLAN1_Policy",
        'state_encoder': None,
        'env_num': 1,
        'sim_num': 1,
        'model_path': "checkpoints/InternVLA-N1",
        'camera_intrinsic': [[585.0, 0.0, 320.0], [0.0, 585.0, 240.0], [0.0, 0.0, 1.0]],
        'width': 640,
        'height': 480,
        'hfov': 79,
        'resize_w': 384,
        'resize_h': 384,
        'max_new_tokens': 1024,
        'num_frames': 32,
        'num_history': 8,
        'num_future_steps': 4,
        'device': 'cuda:0',
        'predict_step_nums': 32,
        'continuous_traj': True,
        # debug
        'vis_debug': True,  # If vis_debug=True, you can get visualization results
        'vis_debug_path': './logs/test/vis_debug',
    },
)

In [ ]:
from internnav.utils import AgentClient

agent = AgentClient(agent_cfg)

/root/InternNav/./internnav/model/encoder/navdp_backbone.py:124: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.rgb_model.load_state_dict(torch.load(checkpoint), strict=

Loading navdp model: NavDP_Policy_DPT_CriticSum_DAT
Pretrained: None
No pretrained weights provided, initializing randomly.


Loading checkpoint shards: 100%|██████████| 4/4 [00:03<00:00,  1.06it/s]
Some weights of the model checkpoint at checkpoints/InternVLA-N1 were not used when initializing InternVLAN1ForCausalLM: ['model.navdp.rgbd_encoder.depth_model.mask_token', 'model.navdp.rgbd_encoder.rgb_model.mask_token']
- This IS expected if you are initializing InternVLAN1ForCausalLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing InternVLAN1ForCausalLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of InternVLAN1ForCausalLM were not initialized from the model checkpoint at checkpoints/InternVLA-N1 and are newly initialized: ['model.navdp.pg_pred_mlp.0.bias', 'model.navdp.pg_pred_mlp.0.weight', 'model.navdp.p

INFO:     ::1:38332 - "POST /agent/init HTTP/1.1" 201 Created


In [ ]:
from scripts.iros_challenge.onsite_competition.sdk.save_obs import load_obs_from_meta
rs_meta_path = '/root/InternNav/scripts/iros_challenge/onsite_competition/captures/rs_meta.json'

fake_obs_640 = load_obs_from_meta(rs_meta_path)
fake_obs_640['instruction'] = 'go to the red car'
print(fake_obs_640['rgb'].shape, fake_obs_640['depth'].shape)

(480, 640, 3) (480, 640)


In [8]:
action = agent.step([fake_obs_640])[0]['action'][0]
print(f"Action taken: {action}")

======== Infer S2 at step 0========


/root/miniconda3/envs/model/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.1` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/root/miniconda3/envs/model/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.001` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/root/miniconda3/envs/model/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:653: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `1` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


============ output 1  ←←←←
s2 infer finish!!
get s2 output lock
=============== [2, 2, 2, 2] =================
Output discretized traj: [2] 0
INFO:     ::1:46114 - "POST /agent/internvla_n1/step HTTP/1.1" 200 OK
Action taken: 2


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
